In [5]:
import pandas as pd
import geopandas as gpd
import os
import getpass
import sys
from pathlib import Path
user = getpass.getuser()
PROJECT_ROOT = Path(f"/Users/{user}/Final_Lawyer_Git July10")
scripts_directory = PROJECT_ROOT / 'Scripts' / 'Helpers'

# Convert the Path object to a string and append it to sys.path
if str(scripts_directory) not in sys.path:
    sys.path.append(str(scripts_directory))
from Helpers import *
# Load data
# FIPS
fips_path = PROJECT_ROOT/"Data/Geography/FIPS/US states FIPS.csv"     # FIPS path
df_fips = pd.read_csv(fips_path)  # Load FIPS
df_fips["FIPS Code"] = df_fips["FIPS Code"].astype(str).str.zfill(2)    # Ensure string, add 0 to single digits

# MSAs
msa_shapefile_path = PROJECT_ROOT/"Data/Geography/CBSA_shapefile_2025/tl_2025_us_cbsa.shp"   # Shapefile path
gdf_msa = gpd.read_file(msa_shapefile_path) # Load MSA shapefile

gdf_msa = gdf_msa[gdf_msa["LSAD"] == 'M1'].reset_index(drop=True)   # Remove micro
gdf_msa['State_Abbr'] = gdf_msa['NAME'].str[-2:]    # Get state abbriviations
gdf_msa = gdf_msa[gdf_msa['State_Abbr'].isin(df_fips['Postal Abbr.'])].copy()   # Filter only US states
valid_geoid = gdf_msa["GEOID"].astype(str).tolist()  # GEOID list

# Inflation
# https://www.officialdata.org/us/inflation/
inflation_factors = {2001: 1.38,
                     2002: 1.36,
                     2003: 1.33,
                     2004: 1.30,
                     2005: 1.26,
                     2006: 1.22,
                     2007: 1.18,
                     2008: 1.14,
                     2009: 1.14,
                     2010: 1.12,
                     2011: 1.09,
                     2012: 1.07,
                     2013: 1.05,
                     2014: 1.04,
                     2015: 1.03,
                     2016: 1.02,
                     2017: 1.00,
                     2018: 0.98,
                     2019: 0.96,
                     2020: 0.95,
                     2021: 0.90,
                     2022: 0.84,
                     2023: 0.80,
                     2024: 0.78} # For 2017

# Population
pop_path = PROJECT_ROOT/"Data/Population Data/MSA Population" # Population data Path
population_data = []    # Data structure

for year in list(range(2010, 2020)) + list(range(2021, 2025)):
    file_name = f"ACSDT1Y{year}.B01003-Data.csv" # Dataset name
    full_path = os.path.join(pop_path, file_name)  # Create path

    df = pd.read_csv(full_path)   # Load data

    df = df[['GEO_ID', 'NAME', 'B01003_001E']].copy()  # Take relevant columns
    df.rename(columns={'GEO_ID': 'AREA', 'NAME': 'MSA', 'B01003_001E': 'Population'}, inplace=True) # Rename columns

    df = df.iloc[1:].reset_index(drop=True) # Remove first row

    df['AREA'] = df['AREA'].str[-5:]    # Get GEOID
    df["AREA"] = df["AREA"].astype(str) # GEOID as string
    df["year"] = year  # Store year

    df = df[df["AREA"].isin(valid_geoid)]   # Take only MSAs from the list

    population_data.append(df) # Add to list

df_population = pd.concat(population_data, ignore_index=True)
df_population['year'] = df_population['year'].astype(int)

# GDP
gdp_path = PROJECT_ROOT/"Data/BLS data/GDP/GDP and Personal Income Formatted.csv"     # GDP path
df_gdp = pd.read_csv(gdp_path)  # Load GDP

year_cols = [str(i) for i in range(2001, 2024)]
df_gdp = df_gdp[['GeoFips', 'GeoName', 'Description', *year_cols]].copy()  # Take relevant columns
df_gdp.rename(columns={'GeoFips': 'AREA', 'GeoName': 'MSA'}, inplace=True)  # Rename columns

df_gdp = df_gdp[df_gdp['Description'] == 'Real GDP (thousands of chained 2017 dollars)']
df_gdp["AREA"] = df_gdp["AREA"].astype(str).str.zfill(5)    # Ensure string, add 0s to three-digits codes
df_gdp[year_cols] = df_gdp[year_cols].apply(pd.to_numeric, errors='coerce') # To numeric
df_gdp = df_gdp[df_gdp["AREA"].isin(valid_geoid)]   # Take only MSAs from the list

df_gdp = df_gdp.melt(id_vars=['AREA', 'MSA'], value_vars=year_cols, var_name='year', value_name='GDP')
df_gdp['year'] = df_gdp['year'].astype(int)
df_gdp = df_gdp.dropna(subset=['GDP'])
df_gdp["GDP"] = df_gdp["GDP"] * 1000    # GDP in thousands of dollars

# Lawyers wages
base_path = PROJECT_ROOT/"Data/Processed Data/Filtered Tables"      # Data path
msa_data = []   # Data structure

for year in range(2005, 2025):
    file_name = f"MSA_{year}_Filtered_Extended_Professions.xlsx" # Dataset name
    full_path = os.path.join(base_path, file_name)  # Create path

    df = pd.read_excel(full_path)   # Load data
    df.columns = df.columns.str.upper()

    df = df[['AREA', 'AREA_TITLE', 'OCC_TITLE', 'TOT_EMP', 'H_MEAN', 'A_MEAN', 'H_MEDIAN', 'A_MEDIAN']].copy()  # Take relevant columns

    df["AREA"] = df["AREA"].astype(str) # GEOID as string
    df["year"] = year  # Store year

    df = df[df["AREA"].isin(valid_geoid)]   # Take only MSAs

    msa_data.append(df) # Add to list

# Build the dataset
all_data = pd.concat(msa_data, ignore_index=True)   # Create a single table
all_data.replace("*", pd.NA, inplace=True)  # Convert "*" to NaN
all_data["H_MEAN"] = pd.to_numeric(all_data["H_MEAN"], errors='coerce') # To numeric
all_data["A_MEAN"] = pd.to_numeric(all_data["A_MEAN"], errors='coerce') # To numeric
all_data["H_MEDIAN"] = pd.to_numeric(all_data["H_MEDIAN"], errors='coerce') # To numeric
all_data["A_MEDIAN"] = pd.to_numeric(all_data["A_MEDIAN"], errors='coerce') # To numeric
all_data["TOT_EMP"] = pd.to_numeric(all_data["TOT_EMP"], errors='coerce') # To numeric
all_data.dropna(subset=["H_MEAN", "A_MEAN", "TOT_EMP"], inplace=True)   # Drop rows with missing values

# Filter lawyers and All Occupations
lawyers_data = all_data[all_data["OCC_TITLE"] == "Lawyers"].copy()
all_occ_data = all_data[all_data["OCC_TITLE"] == "All Occupations"].copy()

# Total expenses
lawyers_data["Legal_Expense"] = lawyers_data["A_MEAN"] * lawyers_data["TOT_EMP"]
all_occ_data["Salary_Expense"] = all_occ_data["A_MEAN"] * all_occ_data["TOT_EMP"]

# Ensure no duplicates
assert lawyers_data.duplicated(["AREA","year"]).sum() == 0
assert all_occ_data.duplicated(["AREA","year"]).sum() == 0

# Merge
combined_data = lawyers_data.merge(
    all_occ_data[['AREA', 'year', 'Salary_Expense', 'A_MEAN']], on=['AREA', 'year'], suffixes=('', '_All'), validate="one_to_one")

# Add Population and GDP
combined_data = combined_data.merge(df_population[['AREA', 'year', 'Population']], on=['AREA', 'year'], how='left')
combined_data = combined_data.merge(df_gdp[['AREA', 'year', 'GDP']], on=['AREA', 'year'], how='left')

# Inflation adjustment
combined_data['inflation_factor'] = combined_data['year'].map(inflation_factors)    # Map factors
assert combined_data['inflation_factor'].notna().all()

combined_data['Legal_Expense'] = combined_data['Legal_Expense'] * combined_data['inflation_factor'] # Convert legal expenses 2017
combined_data['Salary_Expense'] = combined_data['Salary_Expense'] * combined_data['inflation_factor']   # Convert salary expenses 2017
combined_data['A_MEDIAN'] = combined_data['A_MEDIAN'] * combined_data['inflation_factor']   # Convert median annual income 2017
combined_data['A_MEAN'] = combined_data['A_MEAN'] * combined_data['inflation_factor']   # Convert median hourly income 2017

# Clean data
combined_data = combined_data.dropna(subset=['GDP', 'Population', 'Salary_Expense'])
valid_areas = combined_data.groupby("AREA")["year"].nunique()
all_years = combined_data["year"].nunique()
final_data = combined_data[combined_data["AREA"].isin(valid_areas[valid_areas == all_years].index)].copy()  # Final dataset
final_data = final_data.sort_values(["AREA","year"]).reset_index(drop=True) # Sort data

cols = ["TOT_EMP", "Population", "Legal_Expense", "GDP"]    # Columns to convert to numeric
final_data[cols] = final_data[cols].apply(pd.to_numeric, errors='coerce')   # Covert to numeric
final_data = final_data.dropna(subset=cols) # Drop rows with missing data

final_data["lawyers_per_cap"] = final_data["TOT_EMP"] / final_data["Population"] # Per capita lawyers
final_data["legal_per_gdp"] = final_data["Legal_Expense"] / final_data["GDP"] # Per GDP legal expenses
final_data = final_data.rename(columns={'TOT_EMP': 'Num_Lawyers'})  # Reduce ambiguity

# Save
final_data.to_csv(PROJECT_ROOT/"Data/Processed Data/Legal-Economy-Coupling/Legal_Economy_Data.csv", index=False)
